# Natural Language Processing with Disaster Tweets

**Goal:** Predict whether a tweet is about a real disaster (1) or not (0).

**Metric:** F1 score

**Kaggle link:** https://www.kaggle.com/competitions/nlp-getting-started

## 1. Setup & Imports

In [ ]:
# Standard library
import sys
import warnings
from pathlib import Path

# Add project root to Python path so we can import from src/ (ADR-025)
sys.path.insert(0, str(Path("..").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.text import add_text_features

# Settings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

%matplotlib inline

## 2. Configuration

In [ ]:
# Project paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../outputs/models")
SUBMISSIONS_DIR = Path("../outputs/submissions")

# Competition settings
COMPETITION_NAME = "nlp-getting-started"
TARGET_COL = "target"
ID_COL = "id"
TEXT_COL = "text"
RANDOM_STATE = 42
TEST_SIZE = 0.2

## 3. Load Data

In [ ]:
train_df = pd.read_csv(DATA_RAW / "train.csv")
test_df = pd.read_csv(DATA_RAW / "test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Test columns:  {list(test_df.columns)}")
print(f"\nTarget distribution:\n{train_df[TARGET_COL].value_counts()}")
print(f"Target balance (% positive): {train_df[TARGET_COL].mean():.3f}")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Overview
train_df.head()

In [ ]:
# Data types and non-null counts
train_df.info()

In [ ]:
# Statistical summary
train_df.describe()

In [ ]:
# Missing values
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df)) * 100
missing_df = pd.DataFrame({"count": missing, "percentage": missing_pct})
print(missing_df[missing_df["count"] > 0].sort_values("percentage", ascending=False))

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(8, 5))
counts = train_df[TARGET_COL].value_counts()
counts.plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_title(f"Target Distribution: {TARGET_COL}")
ax.set_ylabel("Count")
ax.set_xticklabels(["Not Disaster (0)", "Disaster (1)"], rotation=0)
for i, v in enumerate(counts):
    ax.text(i, v + 50, f"{v} ({v / len(train_df) * 100:.1f}%)", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Text length and word count distributions by target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_df["_text_len"] = train_df[TEXT_COL].str.len()
train_df["_word_count"] = train_df[TEXT_COL].str.split().str.len()

for label in [0, 1]:
    subset = train_df[train_df[TARGET_COL] == label]
    axes[0].hist(subset["_text_len"], bins=50, alpha=0.5, label=f"Target={label}")
axes[0].set_title("Tweet Character Length by Target")
axes[0].set_xlabel("Characters")
axes[0].legend()

for label in [0, 1]:
    subset = train_df[train_df[TARGET_COL] == label]
    axes[1].hist(subset["_word_count"], bins=30, alpha=0.5, label=f"Target={label}")
axes[1].set_title("Tweet Word Count by Target")
axes[1].set_xlabel("Words")
axes[1].legend()

plt.tight_layout()
plt.show()

# Drop temporary EDA columns
train_df.drop(columns=["_text_len", "_word_count"], inplace=True)

In [ ]:
# Keyword analysis
keyword_stats = train_df.groupby("keyword")[TARGET_COL].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(f"Unique keywords: {train_df['keyword'].nunique()}")
print(f"Missing keywords: {train_df['keyword'].isna().sum()} ({train_df['keyword'].isna().mean() * 100:.1f}%)")
print(f"Missing locations: {train_df['location'].isna().sum()} ({train_df['location'].isna().mean() * 100:.1f}%)")
print(f"\nTop 10 disaster keywords:\n{keyword_stats.head(10)}")
print(f"\nTop 10 non-disaster keywords:\n{keyword_stats.tail(10)}")

### EDA Key Findings

1. **Target:** Slightly imbalanced — 57% not disaster, 43% disaster
2. **Text:** Disaster tweets tend to be slightly longer (both character and word count)
3. **Keywords:** 221 unique values, only 0.8% missing. Strong signal — some keywords (derailment, debris, wreckage) are 100% disaster, others (aftershock, body bags, blazing) are near 0%
4. **Location:** 33% missing, 3341 unique values — noisy and high-cardinality, likely low value for a first baseline
5. **Missing values:** Only `keyword` (61) and `location` (2533) have missing values. `text` and `target` are complete
6. **Implication for cleaning:** URL-decode keywords (%20 encoding), handle missing keywords, clean text (URLs, special chars). Drop `location` for baseline

## 5. Data Cleaning

In [ ]:
# Apply text cleaning and add text features to both train and test
# WHY: Same transformations on both sets to avoid train/test mismatch (ADR-019)
train_df = add_text_features(train_df, text_col=TEXT_COL)
test_df = add_text_features(test_df, text_col=TEXT_COL)

print("Sample cleaned text:")
print(train_df[["text", "text_clean"]].head(3).to_string())
print("\nKeyword decoding sample:")
print(train_df[train_df["keyword"].notna()][["keyword", "keyword_clean"]].drop_duplicates().head(5).to_string())
print(f"\nNew columns added: {['text_clean', 'text_len', 'word_count', 'keyword_clean']}")

In [ ]:
# Verification: no unexpected missing values after cleaning
assert train_df["text_clean"].isna().sum() == 0, "text_clean has NaN"
assert test_df["text_clean"].isna().sum() == 0, "test text_clean has NaN"
assert train_df["keyword_clean"].isna().sum() == 0, "keyword_clean has NaN"
assert test_df["keyword_clean"].isna().sum() == 0, "test keyword_clean has NaN"

print(f"Train text_clean: {train_df['text_clean'].isna().sum()} missing")
print(f"Test text_clean:  {test_df['text_clean'].isna().sum()} missing")
print(f"Train keyword_clean: {train_df['keyword_clean'].isna().sum()} missing")
print(f"Test keyword_clean:  {test_df['keyword_clean'].isna().sum()} missing")
print("\nAll clean — no missing values in processed columns.")

## 6. Feature Engineering

In [ ]:
# TODO: Create new features
# Examples:
# train_df["new_feature"] = train_df["col_a"] / train_df["col_b"]
# train_df["is_weekend"] = train_df["day_of_week"].isin([5, 6]).astype(int)

# TODO: Encode categorical variables
# le = LabelEncoder()
# train_df["category_encoded"] = le.fit_transform(train_df["category"])

# TODO: Apply same transformations to test_df!

## 7. Modeling

In [ ]:
# Prepare features and target
# TODO: Select your feature columns
# feature_cols = ["col1", "col2", "col3"]
# X = train_df[feature_cols]
# y = train_df[TARGET_COL]

# Train/validation split
# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
# )
# print(f"Train: {X_train.shape}, Validation: {X_val.shape}")

In [ ]:
# Baseline model
# TODO: Choose your baseline model

# For classification:
# model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
# model.fit(X_train, y_train)

# Cross-validation
# cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
# print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 8. Evaluation

In [ ]:
# Predictions on validation set
# y_pred = model.predict(X_val)

# For classification:
# print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
# print(f"\nClassification Report:")
# print(classification_report(y_val, y_pred))

# Confusion matrix
# fig, ax = plt.subplots(figsize=(8, 6))
# cm = confusion_matrix(y_val, y_pred)
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
# ax.set_title("Confusion Matrix")
# ax.set_ylabel("Actual")
# ax.set_xlabel("Predicted")
# plt.tight_layout()
# plt.show()

In [ ]:
# Feature importance
# TODO: Uncomment after training a tree-based model

# importance_df = pd.DataFrame({
#     "feature": feature_cols,
#     "importance": model.feature_importances_
# }).sort_values("importance", ascending=False)

# fig, ax = plt.subplots(figsize=(10, 6))
# sns.barplot(data=importance_df, x="importance", y="feature", ax=ax)
# ax.set_title("Feature Importance")
# plt.tight_layout()
# plt.show()

## 9. Submission

In [ ]:
# Generate predictions on test set
# TODO: Apply same preprocessing to test_df
# X_test = test_df[feature_cols]
# test_predictions = model.predict(X_test)

# Create submission file
# TODO: Adapt columns to match competition requirements
# submission = pd.DataFrame({
#     "Id": test_df["Id"],
#     TARGET_COL: test_predictions
# })

# submission.to_csv(SUBMISSIONS_DIR / "submission.csv", index=False)
# print(f"Submission saved to {SUBMISSIONS_DIR / 'submission.csv'}")
# print(f"Shape: {submission.shape}")
# submission.head()